# HDLEC vs imLEC genome-wide comparison + ERG cis-regulatory element check
### ATAC-seq chromatin accessibility: HDLEC vs imLEC 

**Aim:** assess how closely the immortalised lymphatic endothelial line (imLEC)
resembles primary lymphatic endothelial cells (HDLEC)to determine whether imLEC is a valid model system for CRISPRi,
specifically focusing on ERG binding and ATAC signal at putative ERG CREs.

**Samples:** HDLEC n=2, imLEC n=2,  All hg38, MACS2 peaks, paired-end.

**Files required:**
- `DiffBind_sampleSheet_HDLEC_imLEC.txt` – DiffBind sample sheet (created in this notebook, Part 1)
- BAM files (blacklist-filtered, indexed) for each HDLEC and imLEC replicate, as listed in the sample sheet
- MACS2 peak calls (`.xls`) for each replicate, as listed in the sample sheet
- `LEC_CREs_labelled.bed` – HDLEC consensus CRE/peak set
- `imlec.replicated_broadPeak.clean.bed` – imLEC replicated broadPeak set
- BigWig files (`*.BPM.bw`) for each replicate, in a `bigwigs/` subdirectory
- `hg38-blacklist.v2.bed` – ENCODE hg38 blacklist file (for deepTools `computeMatrix`)
- `LEC_ERG_q0.001.narrowPeak` – ERG ChIP-seq peak calls (Part 2)
- `ERG_extended_TAD_reg_regions.bed` – putative tiered ERG CRE regions (Part 2)
- `dbObj_counted.rds` – optional cached DiffBind object; if present, skip straight to loading it instead of re-running the counting step in Part 1

**Pipeline Created by Dr Hannah Maude in the Cebola lab. Adapted by Tamara Vujic**

This pipeline uses a mixture of Bash and R code, please read comments in each section before running code

## Requirements

**Conda Environments:**
- `diffbind` – R 4.3.3, built from `diffbind.yml` (DiffBind, GenomicRanges, tidyverse, ChIPseeker, TxDb.Hsapiens.UCSC.hg38.knownGene, org.Hs.eg.db, clusterProfiler, etc.) – used for all R / VS Code sections
- `ATAC` – built from `ATAC.yml` (deepTools, BEDTools) – used for terminal sections (`computeMatrix`, `plotHeatmap`, `intersectBed`)

See the code cell below for the environment creation commands.


In [ ]:
#Prior to running this notebook you will need to create a diffbind conda environment. 
#Paste the comands below into the terminal 
#Diffbind.yml and ATAC.yml files were created by Dr Hannah Maude
#Change paths to your own 
#comment out activation lines as needed


module load miniforge/3
miniforge-setup
eval "$(~/miniforge3/bin/conda shell.bash hook)"
conda env create -p /rds/general/user/tv722/home/anaconda3/envs/diffbind -f /rds/general/project/cebola_lab_general/live/conda_envs/diffbind.yml
conda env create -p /rds/general/user/tv722/home/anaconda3/envs/ATAC -f /rds/general/project/cebola_lab_general/live/conda_envs/ATAC.yml
#conda activate /rds/general/user/tv722/home/anaconda3/envs/diffbind
#conda activate /rds/general/user/tv722/home/anaconda3/envs/ATAC

#To make the diffbind environment compatible with R, in the terminal run
R
IRkernel::installspec(user = TRUE)
IRkernel::installspec(name = 'r-diffbind', displayname = 'R 4.3.3 (r-diffbind)')
quit()

**Diffbind sample sheet**

In [ ]:
# Sample sheet: maps each sample to its BAM and peak file, with metadata
# follow the structure, change according to own samples
# Save this as DiffBind_sampleSheet_HDLEC_imLEC.txt in working directory 

SampleID,Factor,Replicate,bamReads,Peaks,PeakCaller,Tissue,Condition
LEC_rep1,batch1,1,/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imlec_atac_comparison/data/HDLEC/rep1.blacklist-filtered-indexed.bam,/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imlec_atac_comparison/data/HDLEC/rep1_peaks.xls,macs,LEC,control
LEC_rep2,batch1,2,/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imlec_atac_comparison/data/HDLEC/rep2.blacklist-filtered-indexed.bam,/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imlec_atac_comparison/data/HDLEC/rep2_peaks.xls,macs,LEC,control
IMLEC_rep1,batch1,1,/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imLEC_ATACseq/FASTQ/01_0MOH_034BICL_Imlec-ATAC-1_ATAC_hs_i10-28.shifted.sorted.bam,/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imLEC_ATACseq/FASTQ/01_0MOH_034BICL_Imlec-ATAC-1_ATAC_hs_i10-28_peaks.xls,macs,IMLEC,control
IMLEC_rep2,batch1,2,/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imLEC_ATACseq/FASTQ/02_0MOI_034BICL_Imlec-ATAC-2_ATAC_hs_i11-29.shifted.sorted.bam,/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imLEC_ATACseq/FASTQ/02_0MOI_034BICL_Imlec-ATAC-2_ATAC_hs_i11-29_peaks.xls,macs,IMLEC,control

# PART 1: Genome wide comparison


In [ ]:
#In the terminal

#Data sets were generated at different times by different individuals thus there may be differences in accessibility owing to technicalities
#To avoid this we define a subset of highly specific peaks as those with strong signal in one cell type and weak signal in the other, directly 
#quantifying signal from bam files and subset peaks based on quantile thresholds. 
#An initial list of CREs was defined by combining HDLEC CREs with additiona; peaks in imLEC. 

# Convert each file to 3-column format
cut -f 1-3 LEC_CREs_labelled.bed > LEC_3col.bed
cut -f 1-3 imlec.replicated_broadPeak.clean.bed > imlec_3col.bed

# move all HDLEC CREs into new file
cat LEC_3col.bed > combined_CREs.bed

# Add imlec peaks that aren't in LEC
module load BEDTools/2.31.1-GCC-14.3.0
intersectBed -v -a imlec_3col.bed -b LEC_3col.bed | wc -l
#27494
intersectBed -v -a imlec_3col.bed -b LEC_3col.bed >> combined_CREs.bed

# Remove extra chromosomes and sort
grep -ve "random" -ve "chrUn" -ve "chrY" combined_CREs.bed | sort -k 1V,1 -k 2,2n > tmp; mv tmp combined_CREs_HDLEC_imLEC.bed

# Total peaks
wc -l combined_CREs_HDLEC_imLEC.bed
#113664


In [ ]:
##In VS code, select the Diffbind kernel in the top right corner##

#Load Libraries
library(DiffBind)
library(GenomicRanges)
library(tidyverse)
library(matrixStats)
library(GenomeInfoDb)
library(rtracklayer)
library(ChIPseeker)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
library(clusterProfiler)
library(org.Hs.eg.db)
library(tidyr)
library(dplyr)

# Set working directory
setwd("/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imlec_atac_comparison")


In [2]:
##In VS code
# ONCE dbObj has been run the first time and saved, start from here and ignore the next cell.
dbObj <- readRDS("dbObj_counted.rds")

In [ ]:
##In VS code
#This section builds the affinity binding matrix: read counts for every sample over a common set of candidate regulatory elements (CREs), 
#which all downstream comparisons depend on. 

# Read in sample sheet
samples <- read.csv('DiffBind_sampleSheet_HDLEC_imLEC.txt')
samples[,c("SampleID", "Replicate", "Tissue")]

# read in the peaks set column names and convert to GRanges
peaks <- read.delim('combined_CREs_HDLEC_imLEC.bed')
colnames(peaks) <- c('chr','start','end')
peaks.G <- makeGRangesFromDataFrame(peaks)

##Reading in Peaksets
dbObj <- dba(sampleSheet='DiffBind_sampleSheet_HDLEC_imLEC.txt')

##Affinity binding matrix
# Remove ENCODE blacklist regions (artefactually high signal)
dbObj <- dba.blacklist(dbObj) 
# Count reads. summits=100 recenters each region on its summit and takes +/-100bp,
# so the comparison is on peak-centred signal rather than whole called peaks
dbObj <- dba.count(dbObj, bUseSummarizeOverlaps=TRUE, summits=100, peaks=peaks.G)

#########
### If run keeps failing, run each file separately###
#si <- Seqinfo(genome = "hg38")
#gr <- dbObj$binding  # or dbObj$peaks[[1]] depending on state

#seqlevels(peaks.G) <- seqlevels(si)[match(seqlevels(peaks.G), seqlevels(si))]
#seqinfo(peaks.G) <- si[seqlevels(peaks.G)]
#peaks.G <- trim(peaks.G)
#sum(start(peaks.G) < 101)   # would go negative after recentering

#dbObj$config$RunParallel <- FALSE
#dbObj <- dba.count(dbObj, bUseSummarizeOverlaps = TRUE, summits = 100, peaks = peaks.G)
#########

# Check: FRiP (fraction of reads in peaks) is the key QC metric. >0.2 is good for
# ATAC-seq; low values indicate poor signal-to-noise
dbObj
dim(dba.peakset(dbObj, bRetrieve = TRUE))

# Save dataset so you don't need to re run again
saveRDS(dbObj, "dbObj_counted.rds")



In [ ]:
##In VS code
#Investigate correlation for the samples 

dba.plotPCA(dbObj, DBA_TISSUE, label = DBA_ID)

dbObj <- dba.normalize(dbObj, normalize = DBA_NORM_RLE)
dba.counts <- dba.peakset(dbObj, bRetrieve = TRUE)
dba.counts.matrix <- mcols(dba.counts) %>% as.matrix

# Replicate correlation QC check
cor(dba.counts.matrix[, "LEC_rep1"], dba.counts.matrix[, "LEC_rep2"], method = "spearman")
cor(dba.counts.matrix[, "IMLEC_rep1"], dba.counts.matrix[, "IMLEC_rep2"], method = "spearman")

# check normalisation worked - column sums should be comparable, within 10%
dba.counts.matrix %>% colSums %>% enframe

In [ ]:
##In VS code

# Normalise
# RLE normalisation. Size factors are derived from the count matrix itself,
# relative to a reference built from the geometric mean across all samples.
dbObj <- dba.normalize(dbObj, normalize = DBA_NORM_RLE)

# Extract the normalised binding matrix as a GRanges with counts in mcols
dba.counts <- dba.peakset(dbObj, bRetrieve = TRUE)
dba.counts.matrix <- mcols(dba.counts) %>% as.matrix

# check normalisation worked - column sums should be comparable, within 10%
dba.counts.matrix %>% colSums %>% enframe

In [6]:
##In VS code

# Define the quantiles by peak signal
#Top LEC 25%
lec75 <- quantile(dba.counts.matrix[,1:2])[4]
#LEC 50%
lec50 <- quantile(dba.counts.matrix[,1:2])[3]
#Bottom LEC 25%
lec25 <- quantile(dba.counts.matrix[,1:2])[2]

#Top IMLEC 25%
imlec75 <- quantile(dba.counts.matrix[,3:4])[4]
#IMLEC 50%
imlec50 <- quantile(dba.counts.matrix[,3:4])[3]
#Bottom IMLEC 25%
imlec25 <- quantile(dba.counts.matrix[,3:4])[2]

In [7]:
##In VS code
# Calculate the median counts across replicates
dba.counts.matrix <- dba.counts.matrix %>% as.data.frame %>% 
    rowwise %>% 
    mutate(LEC_median = median(c_across(c(LEC_rep1, LEC_rep2))),
           imLEC_median = median(c_across(c(IMLEC_rep1, IMLEC_rep2))))

In [ ]:
##In VS code

# Reads were counted over a consensus CRE set using DiffBind (v3.12.0) with 100 bp summit-centred intervals and normalised by RLE. 
# Median normalised counts were calculated per cell type across replicates. Cell-type-enriched CREs were defined as regions above the 
# median in one cell type and below the 25th percentile in the other; because the lower quartile of the count distribution approaches zero, 
# these represent regions accessible in one cell type and largely inaccessible in the other.


# Plot the median HDLEC counts against the median imLEC counts and highlight regions outside the quantiles

# LEC vs IMLEC
LECvsIMLECplot <- ggplot(dba.counts.matrix, aes(x = LEC_median, y =  imLEC_median), pch=20) +
geom_point(col='grey', alpha = 0.5) + 
theme_minimal() + 
geom_point(data = dba.counts.matrix %>% subset(LEC_median > lec50 &  imLEC_median < imlec25), 
               aes(x = LEC_median, y =  imLEC_median), col = "purple") + 
geom_point(data = dba.counts.matrix %>% subset(LEC_median < lec25 &  imLEC_median > imlec50), 
               aes(x = LEC_median, y =  imLEC_median), col = "blue") +
xlab("HDLEC normalised counts") +
ylab("IMLEC normalised counts")+
  ggtitle("Genome-wide accessibility: HDLEC vs imLEC")

  LECvsIMLECplot

## LECvsIMLECplot ##
# Median normalised ATAC-seq counts across the HDLEC+imLEC consensus CRE
# set. Each point is one CRE; grey points show all regions. Blue points
# mark CREs accessible in HDLEC but closed in imLEC (lost on
# immortalisation); purple points mark CREs gained in imLEC relative to
# HDLEC.

#ggsave(LECvsIMLECplot, file = "HDLEC_vs_imLEC_genomewide_scatter.pdf", width = 6, height = 5, dpi = 500)

# cell specific signal 
dba.counts.matrix %>% subset(LEC_median   > lec50   & imLEC_median < imlec25) %>% nrow
#6272
dba.counts.matrix %>% subset(LEC_median   < lec25   & imLEC_median > imlec50) %>% nrow
#6573

In [ ]:
##In VS code

# Simplified classification: HDLEC-specific, imLEC-specific, and
# everything else into one Shared category. 

topLEC_vsIMLEC <- dba.counts[which(dba.counts.matrix$LEC_median   > lec50   &
                                    dba.counts.matrix$imLEC_median < imlec25),]
topIMLEC_vsLEC <- dba.counts[which(dba.counts.matrix$imLEC_median > imlec50 &
                                    dba.counts.matrix$LEC_median   < lec25),]

specific_idx <- unique(c(
  which(dba.counts.matrix$LEC_median   > lec50   & dba.counts.matrix$imLEC_median < imlec25),
  which(dba.counts.matrix$imLEC_median > imlec50 & dba.counts.matrix$LEC_median   < lec25)
))
shared_idx <- setdiff(seq_len(nrow(dba.counts.matrix)), specific_idx)
shared_HDLEC_imLEC <- dba.counts[shared_idx,]

counts_summary <- sapply(list(HDLEC_specific = topLEC_vsIMLEC,
                               imLEC_specific = topIMLEC_vsLEC,
                               shared         = shared_HDLEC_imLEC), length)

print(counts_summary)
cat(sprintf("\nSum of all categories: %d\n", sum(counts_summary)))
cat(sprintf("Total consensus CREs (dba.counts): %d\n", length(dba.counts)))
stopifnot(sum(counts_summary) == length(dba.counts))

export.bed(topLEC_vsIMLEC,     con = 'topLEC_vsIMLEC.bed')
export.bed(topIMLEC_vsLEC,     con = 'topIMLEC_vsLEC.bed')
export.bed(shared_HDLEC_imLEC, con = 'shared_HDLEC_imLEC.bed')

# Subsample the shared set for heatmap display only, 25000 so it still looks proportional 
set.seed(1)
n_shared <- length(shared_HDLEC_imLEC)
export.bed(shared_HDLEC_imLEC[sample(n_shared, min(25000, n_shared))],
con = 'shared_HDLEC_imLEC_25000.bed')

In [ ]:
##In VS code

# Annotate regions using ChIPseeker to see what the shared and cell specific regions consist of 
annoLEC_vsIMLEC <- annotatePeak(topLEC_vsIMLEC, tssRegion = c(-3000, 3000), TxDb = txdb, annoDb = "org.Hs.eg.db")
annoIMLEC_vsLEC <- annotatePeak(topIMLEC_vsLEC, tssRegion = c(-3000, 3000), TxDb = txdb, annoDb = "org.Hs.eg.db")
annoShared      <- annotatePeak(shared_HDLEC_imLEC, tssRegion = c(-3000, 3000), TxDb = txdb, annoDb = "org.Hs.eg.db")

listanno <- list(annoLEC_vsIMLEC, annoIMLEC_vsLEC, annoShared)
names(listanno) <- c(paste0('HDLEC-specific (n=', length(topLEC_vsIMLEC), ')'),
                      paste0('imLEC-specific (n=', length(topIMLEC_vsLEC), ')'),
                      paste0('Shared (n=', length(shared_HDLEC_imLEC), ')'))
plotAnnoBar(listanno)


#### Visualise signal using deeptools heatmaps ###

**in the terminal**:
Create a script called computeMatrix_HDLEC_imLEC_identity.pbs, editing the paths and samples accordingly 
submit in the terminal using qsub computeMatrix_HDLEC_imLEC_identity.pbs



In [ ]:
#PBS -l select=1:mem=60gb:ncpus=16
#PBS -l walltime=02:00:00
#PBS -N computeMatrix_HDLEC_imLEC_identity

eval "$(~/miniforge3/bin/conda shell.bash hook)"
conda activate /rds/general/user/tv722/home/anaconda3/envs/ATAC

DIR=/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imlec_atac_comparison
REFDIR=/rds/general/project/cebola_lab_general/live/reference-genomes/GRCh38

LEC1=$DIR/bigwigs/LEC_rep1.BPM.bw
LEC2=$DIR/bigwigs/LEC_rep2.BPM.bw
IML1=$DIR/bigwigs/imLEC_rep1.BPM.bw
IML2=$DIR/bigwigs/imLEC_rep2.BPM.bw

# Genome-wide HDLEC vs imLEC identity: regions specific to each cell type
computeMatrix reference-point --referencePoint center -p 16 -S $LEC1 $LEC2 $IML1 $IML2 -R $DIR/shared_HDLEC_imLEC_25000.bed $DIR/topLEC_vsIMLEC.bed $DIR/topIMLEC_vsLEC.bed -bs 100 -b 2000 -a 2000 -o $DIR/HDLEC_imLEC_identity_shared_first.matrix.gz --blackListFileName $REFDIR/hg38-blacklist.v2.bed

In [ ]:
#in the Terminal 
#This will plot all the replicates into a heat map 

plotHeatmap -m $DIR/HDLEC_imLEC_identity_shared_first.matrix.gz \
    --outFileName $DIR/final_replicate_HDLEC_imLEC_identity_2kb.pdf \
    --heatmapHeight 25 \
    --samplesLabel HDLEC1 HDLEC2 imLEC1 imLEC2 \
    --refPointLabel center \
    --sortRegions no \
    --colorList "white,#61AA81" "white,#61AA81" "white,orange" "white,orange" \
    --missingDataColor 1 \
    --plotTitle "HDLEC vs imLEC chromatin identity" \
    --legendLocation upper-left


#### Plot median signal across donors
To obtain a summary heatmap for the main figure, we use a custom R script to calculate the median values across the donors for each 100bp bin.
The terminal and R are both used for this section


In [ ]:
#in the Terminal 

# Decompress the matrix file for reading into R
gunzip -c HDLEC_imLEC_identity_shared_first.matrix.gz > HDLEC_imLEC_identity_shared_first.matrix

# Inspect the current header before editing it:
head -1 HDLEC_imLEC_identity_shared_first.matrix   # inspect group_boundaries / sample_boundaries

# The first line of a deepTools matrix file is a metadata header - a JSON
# blob starting with @ that tells plotHeatmap how to interpret the numbers
# below it. The data itself is just a table with no column labels, so the
# header is the only place that records which columns belong to which
# sample (e.g. "columns 7-46 are sample one, 47-86 are sample two").
#
# The R code below collapses 4 samples (LEC_rep1, LEC_rep2, IMLEC_rep1,
# IMLEC_rep2) down to 2 medians (HDLEC, imLEC), so the data table changes
# shape - but R doesn't write a header (col.names = FALSE), so a new one
# has to be supplied by hand. Compare against the header printed above:
#   - "sample_boundaries": drop from 4 samples to 2 (e.g. [0,40,80,120,160]
#     -> [0,40,80]), since each sample still spans 40 bins
#   - "sample_labels": rename to match (e.g. ["HDLEC","imLEC"])
#   - "group_boundaries" / "group_labels": unchanged - there are still 3
#     region groups (shared, topLEC_vsIMLEC, topIMLEC_vsLEC); only the
#     per-sample columns have changed

#Replace header
cat > HDLEC_imLEC_header.txt << 'EOF'
@{"upstream":[2000,2000],"downstream":[2000,2000],"body":[0,0],"bin size":[100,100],"ref point":["center","center"],"verbose":false,"bin avg type":"mean","missing data as zero":false,"min threshold":null,"max threshold":null,"scale":1,"skip zeros":false,"nan after end":false,"proc number":16,"sort regions":"keep","sort using":"mean","unscaled 5 prime":[0,0],"unscaled 3 prime":[0,0],"group_labels":["shared_HDLEC_imLEC_25000.bed","topLEC_vsIMLEC.bed","topIMLEC_vsLEC.bed"],"group_boundaries":[0,25000,31272,37845],"sample_labels":["HDLEC","imLEC"],"sample_boundaries":[0,40,80]}
EOF

In [ ]:
## In VS code

# Read in the matrix
mat <- read.delim('HDLEC_imLEC_identity_shared_first.matrix', skip = 1, header = FALSE)
ncol(mat)   # 166
nrow(mat)   # 37845

# Make a new dataframe for the median values
# Carry over the first six bed file columns
med <- mat[, 1:6]

# Fill in the columns for two samples (80 total bins) with NA values
med[, 7:86] <- NA

for (x in 7:46)  med[, x] <- apply(mat[, c(x, x + 40)], 1, median)     # HDLEC1,HDLEC2 -> HDLEC
for (x in 47:86) med[, x] <- apply(mat[, c(x + 40, x + 80)], 1, median) # imLEC1,imLEC2 -> imLEC
write.table(med, 'HDLEC_imLEC_identity_shared_first_BPM_median.matrix', quote = FALSE, sep = '\t', row.names = FALSE, col.names = FALSE)


In [ ]:
# In the Terminal 

# Replace R's NA missing values with NaN, which is what deeptools expects
sed -i 's/NA/NaN/g' HDLEC_imLEC_identity_shared_first_BPM_median.matrix
# Prepend the edited metadata header describing the new 3-sample layout
cat HDLEC_imLEC_header.txt HDLEC_imLEC_identity_shared_first_BPM_median.matrix > HDLEC_imLEC_median_final.matrix
# Compress to .matrix.gz, the format plotHeatmap reads
gzip HDLEC_imLEC_median_final.matrix
# Plot heatmap
plotHeatmap -m HDLEC_imLEC_median_final.matrix.gz \
     --outFileName final_HDLEC_imLEC_identity_median.pdf \
     --heatmapHeight 20 --heatmapWidth 4 \
     --samplesLabel HDLEC imLEC \
     --refPointLabel center --sortRegions no \
     --colorList "white,#61AA81" "white,orange" \
     --missingDataColor 1 \
     --plotTitle "imLEC accessibility at HDLEC-defined identity CREs" \
     --legendLocation upper-left

# PART 2: ERG-specific comparison

In [ ]:
## in VS code

# Load ERG ChIP peaks and each cell type's own ATAC peak calls

erg_narrowpeak <- read.delim("LEC_ERG_q0.001.narrowPeak", header = FALSE)
colnames(erg_narrowpeak)[1:3] <- c("chr", "start", "end")
erg_chip <- makeGRangesFromDataFrame(erg_narrowpeak)
length(erg_chip)   # 24,683

# Each cell type's own called ATAC peaks (peak calls, not DiffBind
# quantile-classified regions - this is a simple, direct "was a peak
# called here" comparison)
hdlec_atac <- import.bed("LEC_CREs_labelled.bed")
imlec_atac <- import.bed("imlec.replicated_broadPeak.clean.bed")

# Clean 3-column BED of the raw ERG ChIP peaks, for use with computeMatrix
write.table(erg_narrowpeak[, 1:3], "LEC_ERG_3col.bed",
            quote = FALSE, sep = "\t", row.names = FALSE, col.names = FALSE)



In [ ]:
## in VS code

# ERG ChIP data is first intersected with HDLEC ATAC to keep ERG bound regions that have accessible ATAC signal 
# This is then comapred to imLEC ATAC

erg_open_bound_HDLEC <- erg_chip[overlapsAny(erg_chip, hdlec_atac)]

cat("\n functional (open + bound) HDLEC ERG CREs\n")
cat("  ERG ChIP peaks overlapping HDLEC open chromatin:", length(erg_open_bound_HDLEC),
    sprintf("(%.1f%% of raw ChIP peaks)", 100 * length(erg_open_bound_HDLEC) / length(erg_chip)), "\n")

export.bed(erg_open_bound_HDLEC, con = "ERG_CREs_open_and_bound_HDLEC.bed")

erg_accessible_imLEC <- overlapsAny(erg_open_bound_HDLEC, imlec_atac)

cat("  Retained in imLEC: ", sum(erg_accessible_imLEC),
    sprintf("(%.1f%%)", 100 * mean(erg_accessible_imLEC)), "\n")
cat("  Lost in imLEC:     ", sum(!erg_accessible_imLEC),
    sprintf("(%.1f%%)", 100 * mean(!erg_accessible_imLEC)), "\n")

erg_retained <- erg_open_bound_HDLEC[erg_accessible_imLEC]
erg_lost    <- erg_open_bound_HDLEC[!erg_accessible_imLEC]

export.bed(erg_retained, con = "ERG_CREs_functional_retained_in_imLEC.bed")
export.bed(erg_lost,     con = "ERG_CREs_functional_lost_in_imLEC.bed")

In [ ]:
## in VS code

# Which regions are near the ERG CREs imLEC loses?
# retained vs lost ERG CREs? (promoter / intron / intergenic etc.)
# Same ChIPseeker approach as the genome-wide section, applied to the retained/lost set. 


annoRetained <- annotatePeak(erg_retained, tssRegion = c(-3000, 3000), TxDb = txdb, annoDb = "org.Hs.eg.db")
annoLost     <- annotatePeak(erg_lost,     tssRegion = c(-3000, 3000), TxDb = txdb, annoDb = "org.Hs.eg.db")

listanno <- list(annoRetained, annoLost)
names(listanno) <- c(
  paste0('Retained in imLEC (n=', length(erg_retained), ')'),
  paste0('Lost in imLEC (n=', length(erg_lost), ')')
)
plotAnnoBar(listanno) + ggtitle("functional (ChIP + HDLEC ATAC) ERG CREs vs imLEC ATAC")

# ggsave calls if needed:
# ggsave(file = "ERG_feature_distribution.pdf", width = 8, height = 4, dpi = 500)

In [ ]:
## in VS code

# Compare lost ERG-binding sites against the putative tiered ERG CRE file

putative_cres <- import.bed("ERG_extended_TAD_reg_regions.bed")

overlap_hits <- overlapsAny(erg_lost, putative_cres)

cat(sprintf("Lost ERG-binding sites: %d total\n", length(erg_lost)))
cat(sprintf("Overlapping a putative CRE: %d (%.1f%%)\n",
            sum(overlap_hits), 100 * mean(overlap_hits)))

# The actual overlapping regions, for direct inspection
lost_ERG_matching_putative <- erg_lost[overlap_hits]
export.bed(lost_ERG_matching_putative, con = "lost_ERG_binding_overlapping_putative_CREs.bed")

# Which putative CREs (and which tier) are hit by a lost ERG-binding site
putative_hit <- overlapsAny(putative_cres, erg_lost)
cat(sprintf("\nPutative CREs overlapping a lost ERG-binding site: %d of %d\n",
            sum(putative_hit), length(putative_cres)))

putative_cres[putative_hit]

##### Visualise signal using deeptools heatmaps using ERG focussed data

**in the terminal**:
Create a script called computeMatrix_ERG_functional_HDLEC_imLEC.pbs, editing the paths and samples accordingly 
submit in the terminal using qsub computeMatrix_ERG_functional_HDLEC_imLEC.pbs

In [ ]:
#PBS -l select=1:mem=40gb:ncpus=16
#PBS -l walltime=01:00:00
#PBS -N computeMatrix_ERG_functional_HDLEC_imLEC

eval "$(~/miniforge3/bin/conda shell.bash hook)"
conda activate /rds/general/user/tv722/home/anaconda3/envs/ATAC

DIR=/rds/general/user/tv722/projects/erg_lymphangiogenesis_project/live/ECs_ATAC/imlec_atac_comparison
REFDIR=/rds/general/project/cebola_lab_general/live/reference-genomes/GRCh38


LEC1=$DIR/bigwigs/LEC_rep1.BPM.bw
LEC2=$DIR/bigwigs/LEC_rep2.BPM.bw
IML1=$DIR/bigwigs/imLEC_rep1.BPM.bw
IML2=$DIR/bigwigs/imLEC_rep2.BPM.bw

computeMatrix reference-point --referencePoint center -p 16 \
    -S $LEC1 $LEC2 $IML1 $IML2 \
    -R $DIR/ERG_CREs_functional_retained_in_imLEC.bed $DIR/ERG_CREs_functional_lost_in_imLEC.bed \
    -bs 50 -b 1000 -a 1000 \
    -o $DIR/ERG_functional_HDLEC_imLEC.matrix.gz \
    --blackListFileName $REFDIR/hg38-blacklist.v2.bed \
    --sortRegions keep

In [ ]:
#in the Terminal 
#This will plot all the replicates into a heat map 
plotHeatmap -m $DIR/ERG_functional_HDLEC_imLEC.matrix.gz \
    --outFileName $DIR/final_ERG_functional_HDLEC_imLEC.pdf \
    --heatmapHeight 25 \
    --samplesLabel HDLEC1 HDLEC2 imLEC1 imLEC2 \
    --refPointLabel center \
    --sortRegions keep \
    --regionsLabel  "Retained in imLEC" "Lost in imLEC" \
    --colorList "white,#B03060" "white,#B03060" "white,#2E8B8B" "white,#2E8B8B" \
    --missingDataColor 1 \
    --plotTitle "ATAC accessibility at functional ERG CREs, HDLEC vs imLEC" \
    --legendLocation upper-left

#### Plot median signal across donors
To obtain a summary heatmap for the main figure, we use a custom R script to calculate the median values across the donors for each 100bp bin.
The terminal and R are both used for this section


In [ ]:
#in the Terminal 
# Decompress the matrix file for reading into R
gunzip -c ERG_functional_HDLEC_imLEC.matrix.gz > ERG_functional_HDLEC_imLEC.matrix

# Inspect the current header before editing it:
head -1 ERG_functional_HDLEC_imLEC.matrix

# The first line of a deepTools matrix file is a metadata header - a JSON
# blob starting with @ that tells plotHeatmap how to interpret the numbers
# below it. The data itself is just a table with no column labels, so the
# header is the only place that records which columns belong to which
# sample (e.g. "columns 7-46 are sample one, 47-86 are sample two").
#
# The R code below collapses 4 samples (LEC_rep1, LEC_rep2, IMLEC_rep1,
# IMLEC_rep2) down to 2 medians (HDLEC, imLEC), so the data table changes
# shape - but R doesn't write a header (col.names = FALSE), so a new one
# has to be supplied by hand. Compare against the header printed above:
#   - "sample_boundaries": drop from 4 samples to 2 (e.g. [0,40,80,120,160]
#     -> [0,40,80]), since each sample still spans 40 bins
#   - "sample_labels": rename to match (e.g. ["HDLEC","imLEC"])
#   - "group_boundaries" / "group_labels": unchanged - the regions (rows,
#     Retained/Lost) haven't changed, only the per-sample columns have


#Replace header
cat > ERG_functional_median_header.txt << 'EOF'
@{"upstream":[1000,1000],"downstream":[1000,1000],"body":[0,0],"bin size":[50,50],"ref point":["center","center"],"verbose":false,"bin avg type":"mean","missing data as zero":false,"min threshold":null,"max threshold":null,"scale":1,"skip zeros":false,"nan after end":false,"proc number":16,"sort regions":"keep","sort using":"mean","unscaled 5 prime":[0,0],"unscaled 3 prime":[0,0],"group_labels":["ERG_CREs_functional_retained_in_imLEC.bed","ERG_CREs_functional_lost_in_imLEC.bed"],"group_boundaries":[0,17582,21417],"sample_labels":["HDLEC","imLEC"],"sample_boundaries":[0,40,80]}
EOF

In [ ]:
## in VS code

mat <- read.delim('ERG_functional_HDLEC_imLEC.matrix', skip = 1, header = FALSE)
ncol(mat)   # 166
nrow(mat)   # 21417

med <- mat[, 1:6]
med[, 7:86] <- NA
 
# Columns 7:46 = LEC_rep1, 47:86 = LEC_rep2 -> median = HDLEC
# Columns 87:126 = imLEC_rep1, 127:166 = imLEC_rep2 -> median = imLEC
for (x in 7:46)  med[, x] <- apply(mat[, c(x, x + 40)], 1, median)
for (x in 47:86) med[, x] <- apply(mat[, c(x + 40, x + 80)], 1, median)
 
write.table(med, 'ERG_functional_HDLEC_imLEC_median.matrix',
            quote = FALSE, sep = '\t', row.names = FALSE, col.names = FALSE)


In [ ]:
# In the Terminal 
sed -i 's/NA/NaN/g' ERG_functional_HDLEC_imLEC_median.matrix
cat ERG_functional_median_header.txt ERG_functional_HDLEC_imLEC_median.matrix > ERG_functional_median_final.matrix
gzip ERG_functional_median_final.matrix


plotHeatmap -m ERG_functional_median_final.matrix.gz --outFileName ERG_functional_median_heatmap.pdf --heatmapHeight 20 --heatmapWidth 4 --samplesLabel HDLEC imLEC --refPointLabel center --sortRegions no --regionsLabel "Retained" "Lost" --colorList "white,#61AA81" "white,orange" --missingDataColor 1 --plotTitle "ERG-bound CREs: HDLEC vs imLEC" --legendLocation upper-left
